<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/02_no_supervisado/20_dbscan_mean_shift.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# DBSCAN y Mean-Shift

**Pregunta guía:** ¿Cómo hallamos estructuras sin fijar el número de grupos?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## Densidad y modos

DBSCAN declara núcleo a un punto con al menos `min_samples` dentro de una
bola de radio $\varepsilon$; conecta núcleos y marca como ruido lo no
alcanzable. Mean-Shift asciende hacia modos de una densidad kernel:
$m(x)=\sum_i K_h(x_i-x)x_i/\sum_i K_h(x_i-x)-x$.

Ambos dependen de escala. DBSCAN detecta formas no convexas y ruido;
Mean-Shift encuentra modos pero puede ser costoso y muy sensible al
ancho de banda.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, MeanShift, estimate_bandwidth
from sklearn.datasets import make_moons
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

SEMILLA = 42
X, y_real = make_moons(n_samples=700, noise=0.085, random_state=SEMILLA)
rng = np.random.default_rng(SEMILLA)
ruido = rng.uniform(low=[-1.5, -1.0], high=[2.5, 1.5], size=(55, 2))
X = np.vstack([X, ruido])
y_real = np.r_[y_real, np.full(len(ruido), -1)]
Xs = StandardScaler().fit_transform(X)

vecinos = NearestNeighbors(n_neighbors=6).fit(Xs)
distancias, _ = vecinos.kneighbors(Xs)
kdist = np.sort(distancias[:, -1])
plt.plot(kdist)
plt.ylabel("distancia al 6.º vecino")
plt.xlabel("puntos ordenados")
plt.title("Ayuda visual para elegir epsilon")
plt.show()


In [ ]:
filas = []
for eps in np.linspace(0.10, 0.40, 13):
    etiquetas = DBSCAN(eps=eps, min_samples=6).fit_predict(Xs)
    máscara = etiquetas != -1
    n_grupos = len(set(etiquetas)) - (-1 in etiquetas)
    sil = silhouette_score(Xs[máscara], etiquetas[máscara]) if n_grupos > 1 else np.nan
    filas.append(
        {
            "eps": eps,
            "grupos": n_grupos,
            "ruido": (~máscara).mean(),
            "silhouette_sin_ruido": sil,
        }
    )
tabla = pd.DataFrame(filas)
display(tabla)
mejor_eps = tabla.query("grupos >= 2").sort_values("silhouette_sin_ruido").iloc[-1]["eps"]
etiquetas_db = DBSCAN(eps=mejor_eps, min_samples=6).fit_predict(Xs)


In [ ]:
ancho = estimate_bandwidth(Xs, quantile=0.18, n_samples=500, random_state=SEMILLA)
etiquetas_ms = MeanShift(bandwidth=ancho, bin_seeding=True).fit_predict(Xs)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, etiquetas, título in [
    (axes[0], y_real, "generación conocida (sólo diagnóstico)"),
    (axes[1], etiquetas_db, f"DBSCAN eps={mejor_eps:.2f}"),
    (axes[2], etiquetas_ms, f"Mean-Shift h={ancho:.2f}"),
]:
    ax.scatter(Xs[:, 0], Xs[:, 1], c=etiquetas, cmap="tab10", s=14)
    ax.set_title(título)
plt.tight_layout()
plt.show()
print("ARI DBSCAN:", adjusted_rand_score(y_real, etiquetas_db))
print("ARI Mean-Shift:", adjusted_rand_score(y_real, etiquetas_ms))


ARI usa las etiquetas generadoras y sólo está disponible porque es una
simulación; silhouette no conoce la verdad, pero tampoco decide qué
agrupación tiene significado físico.

**Ejercicios:** cambie unidades de un eje; aumente ruido; estudie la
estabilidad de cada grupo; explique por qué un único `eps` falla cuando
la densidad cambia mucho entre regiones.
